### Import Pandas, Load File and list some rows

In [1]:
import pandas as pd

# Read the Excel file from the specified folder
df = pd.read_csv(r'C:\JV_Portfolio_TestFiles\Nashville Housing Data for Data Cleaning.csv')

# Display the first few rows
df.head()

,UniqueID,ParcelID,LandUse,PropertyAddress,SaleDate,SalePrice,LegalReference,SoldAsVacant,OwnerName,OwnerAddress,Acreage,TaxDistrict,LandValue,BuildingValue,TotalValue,YearBuilt,Bedrooms,FullBath,HalfBath
0,2045,007 00 0 125.00,SINGLE FAMILY,"1808 FOX CHASE DR, GOODLETTSVILLE","April 9, 2013",240000,20130412-0036474,No,"FRAZIER, CYRENTHA LYNETTE","1808 FOX CHASE DR, GOODLETTSVILLE, TN",2.3,GENERAL SERVICES DISTRICT,50000.0,168200.0,235700.0,1986.0,3.0,3.0,0.0
1,16918,007 00 0 130.00,SINGLE FAMILY,"1832 FOX CHASE DR, GOODLETTSVILLE","June 10, 2014",366000,20140619-0053768,No,"BONER, CHARLES & LESLIE","1832 FOX CHASE DR, GOODLETTSVILLE, TN",3.5,GENERAL SERVICES DISTRICT,50000.0,264100.0,319000.0,1998.0,3.0,3.0,2.0
2,54582,007 00 0 138.00,SINGLE FAMILY,"1864 FOX CHASE DR, GOODLETTSVILLE","September 26, 2016",435000,20160927-0101718,No,"WILSON, JAMES E. & JOANNE","1864 FOX CHASE DR, GOODLETTSVILLE, TN",2.9,GENERAL SERVICES DISTRICT,50000.0,216200.0,298000.0,1987.0,4.0,3.0,0.0
3,43070,007 00 0 143.00,SINGLE FAMILY,"1853 FOX CHASE DR, GOODLETTSVILLE","January 29, 2016",255000,20160129-0008913,No,"BAKER, JAY K. & SUSAN E.","1853 FOX CHASE DR, GOODLETTSVILLE, TN",2.6,GENERAL SERVICES DISTRICT,50000.0,147300.0,197300.0,1985.0,3.0,3.0,0.0
4,22714,007 00 0 149.00,SINGLE FAMILY,"1829 FOX CHASE DR, GOODLETTSVILLE","October 10, 2014",278000,20141015-0095255,No,"POST, CHRISTOPHER M. & SAMANTHA C.","1829 FOX CHASE DR, GOODLETTSVILLE, TN",2.0,GENERAL SERVICES DISTRICT,50000.0,152300.0,202300.0,1984.0,4.0,3.0,0.0


### Strip whitespace from column names

In [ ]:
df.columns = df.columns.str.strip()

### Convert column to string type

In [ ]:
df["LegalReference"] = df["LegalReference"].astype(str)

### Convert numeric columns to appropriate types

In [ ]:
for col in ["LandValue", "BuildingValue", "TotalValue", "YearBuilt", "Bedrooms", "FullBath", "HalfBath"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

### Display the DataFrame information

In [ ]:
df.info()

### Display unique values in the SoldAsVacant column


In [ ]:
print(df["SoldAsVacant"].unique())

### Replace values in the SoldAsVacant column

In [ ]:
df["SoldAsVacant"] = df["SoldAsVacant"].replace({"N": "No", "Y": "Yes"})

### Display unique values in the SoldAsVacant column

In [ ]:
print(df["SoldAsVacant"].unique())

### Check for missing values in the PropertyAddress column

In [ ]:
print((df["PropertyAddress"].isnull() | (df["PropertyAddress"] == "")).sum())

### Print UniqueID, ParcelID, and PropertyAddress for rows where PropertyAddress is null

In [ ]:
missing_address = df[df["PropertyAddress"].isnull()][["UniqueID", "ParcelID", "PropertyAddress"]]
print(missing_address)

### If PropertyAddress is NaN, copy previous row's PropertyAddress when ParcelID matches and the current UniqueID is different from the previous row's UniqueID.

In [ ]:

for i in range(1, len(df)):
    if pd.isna(df.at[i, "PropertyAddress"]):
        previous_row = df.iloc[i - 1]

        if (
            df.at[i, "ParcelID"] == previous_row["ParcelID"]
            and df.at[i, "UniqueID"] != previous_row["UniqueID"]
            and not pd.isna(previous_row["PropertyAddress"])
        ):
            df.at[i, "PropertyAddress"] = previous_row["PropertyAddress"]

print((df["PropertyAddress"].isnull()).sum())

2


### If PropertyAddress is NaN, copy next row's PropertyAddress when current ParcelID matches next row's ParcelID and the current UniqueID is different from the next row's UniqueID.

In [ ]:

for i in range(len(df) - 1):
    if pd.isna(df.at[i, "PropertyAddress"]):
        next_row = df.iloc[i + 1]

        if (
            df.at[i, "ParcelID"] == next_row["ParcelID"]
            and df.at[i, "UniqueID"] != next_row["UniqueID"]
            and not pd.isna(next_row["PropertyAddress"])
        ):
            df.at[i, "PropertyAddress"] = next_row["PropertyAddress"]

### Split PropertyAddress into street address and city

In [ ]:

split_address = df["PropertyAddress"].str.split(",", n=1, expand=True)
df["PropertySplitAddress"] = split_address[0]
df["PropertySplitCity"] = split_address[1]

print(df[["PropertyAddress", "PropertySplitAddress", "PropertySplitCity"]].head())

### Split OwnerAddress into street, city, and state

In [ ]:

owner_split = df["OwnerAddress"].str.split(",", n=2, expand=True)
df["OwnerSplitAddress"] = owner_split[0]
df["OwnerSplitCity"] = owner_split[1]
df["OwnerSplitState"] = owner_split[2]

print(df[["OwnerAddress", "OwnerSplitAddress", "OwnerSplitCity", "OwnerSplitState"]].head())

### Display the first few rows

In [ ]:

df.head()

### Print the data type of every column

In [ ]:

print(df.dtypes)

### Columns with blanks

In [ ]:

df.isnull().sum()

### Columns Data percentage with nulls greater than zero 

In [ ]:

missing = df.isnull().mean()*100
print(missing[missing>0].sort_values(ascending=False))

### Evaluate duplicated rows

In [ ]:

df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10)